In [1]:
# ============================
# Day 3 | Step 5
# Writer Verification Evaluation
# ============================

import os
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_curve, auc, accuracy_score


# -----------------------------
# Dataset
# -----------------------------
class HSIPairDataset(Dataset):
    def __init__(self, pair_csv, tensor_dir):
        self.df = pd.read_csv(pair_csv)
        self.tensor_dir = tensor_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        A = torch.load(os.path.join(self.tensor_dir, row["tensor_A"]), weights_only=True)
        B = torch.load(os.path.join(self.tensor_dir, row["tensor_B"]), weights_only=True)

        label = torch.tensor(row["same_writer"], dtype=torch.float32)
        return A, B, label


# -----------------------------
# Model
# -----------------------------
class HSI_CNN_LSTM(nn.Module):
    def __init__(self, bands=149, lstm_hidden=128):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.lstm = nn.LSTM(32, lstm_hidden, batch_first=True)
        self.fc = nn.Linear(lstm_hidden, 64)

    def forward(self, x):
        B, bands, H, W = x.shape
        features = []

        for b in range(bands):
            f = self.cnn(x[:, b:b+1])
            features.append(f.view(B, -1))

        seq = torch.stack(features, dim=1)
        lstm_out, _ = self.lstm(seq)

        return self.fc(lstm_out[:, -1])


# -----------------------------
# Paths
# -----------------------------
PAIR_CSV   = r"D:\PEC\HSI_Project\processed\pair_index.csv"
TENSOR_DIR = r"D:\PEC\HSI_Project\tensors\sentences"
MODEL_PATH = r"D:\PEC\HSI_Project\checkpoints\day3_step3\hsi_cnn_lstm_contrastive.pt"

SAVE_DIR = r"D:\PEC\HSI_Project\results\day3_step5\writer_verification"
os.makedirs(SAVE_DIR, exist_ok=True)


# -----------------------------
# Load model
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

model = HSI_CNN_LSTM().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

print("✅ Model loaded for Writer Verification")


# -----------------------------
# DataLoader
# -----------------------------
dataset = HSIPairDataset(PAIR_CSV, TENSOR_DIR)
loader = DataLoader(dataset, batch_size=1, shuffle=False)


# -----------------------------
# Compute distances
# -----------------------------
all_distances = []
all_labels = []

with torch.no_grad():
    for A, B, label in loader:
        A = A.float().to(device)
        B = B.float().to(device)

        emb_A = model(A)
        emb_B = model(B)

        distance = torch.norm(emb_A - emb_B, dim=1)

        all_distances.append(distance.item())
        all_labels.append(label.item())

all_distances = np.array(all_distances)
all_labels = np.array(all_labels)

print("✅ Embeddings & distances computed")


# -----------------------------
# ROC & Threshold
# -----------------------------
fpr, tpr, thresholds = roc_curve(all_labels, -all_distances)
roc_auc = auc(fpr, tpr)

optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

predictions = (-all_distances >= optimal_threshold).astype(int)
accuracy = accuracy_score(all_labels, predictions)

print("\n📌 WRITER VERIFICATION RESULTS")
print(f"Accuracy  : {accuracy*100:.2f}%")
print(f"AUC       : {roc_auc:.4f}")
print(f"Threshold : {optimal_threshold:.4f}")


# -----------------------------
# Save metrics
# -----------------------------
metrics = {
    "task": "Writer Verification",
    "accuracy": float(accuracy),
    "auc": float(roc_auc),
    "optimal_threshold": float(optimal_threshold),
    "total_pairs": int(len(all_labels)),
    "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open(os.path.join(SAVE_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=4)

print("✅ Metrics saved")


# -----------------------------
# Save distances
# -----------------------------
df = pd.DataFrame({
    "distance": all_distances,
    "label": all_labels
})

df.to_csv(os.path.join(SAVE_DIR, "distances_labels.csv"), index=False)
print("✅ Distances saved")


# -----------------------------
# ROC Curve
# -----------------------------
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve – Writer Verification")
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(SAVE_DIR, "roc_curve.png"), dpi=300, bbox_inches="tight")
plt.close()

print("✅ ROC curve saved")


# -----------------------------
# Distance Distribution
# -----------------------------
same = all_distances[all_labels == 1]
diff = all_distances[all_labels == 0]

plt.figure(figsize=(6,5))
plt.hist(same, bins=40, alpha=0.6, label="Same Writer")
plt.hist(diff, bins=40, alpha=0.6, label="Different Writer")
plt.axvline(-optimal_threshold, color='red', linestyle='--', label="Threshold")
plt.xlabel("Embedding Distance")
plt.ylabel("Frequency")
plt.title("Distance Distribution – Writer Verification")
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(SAVE_DIR, "distance_distribution.png"), dpi=300, bbox_inches="tight")
plt.close()

print("✅ Distance distribution saved")


# -----------------------------
# Metadata
# -----------------------------
meta = {
    "experiment": "Day 3 Step 5",
    "task": "Writer Verification",
    "model": "CNN-LSTM + Contrastive Learning",
    "bands": 149,
    "embedding_dim": 64,
    "device": device
}

with open(os.path.join(SAVE_DIR, "metadata.json"), "w") as f:
    json.dump(meta, f, indent=4)

print("\n🎉 Writer Verification Evaluation Completed")
print(f"📂 Saved in: {SAVE_DIR}")


C:\Users\dell\AppData\Local\Temp\ipykernel_16784\1117510188.py:89: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location=de

✅ Model loaded for Writer Verification
✅ Embeddings & distances computed

📌 WRITER VERIFICATION RESULTS
Accuracy  : 65.10%
AUC       : 0.7056
Threshold : -0.5423
✅ Metrics saved
✅ Distances saved
✅ ROC curve saved
✅ Distance distribution saved

🎉 Writer Verification Evaluation Completed
📂 Saved in: D:\PEC\HSI_Project\results\day3_step5\writer_verification


In [3]:
# =========================================
# Day 3 | Step 5: Ink Mismatch Evaluation
# =========================================

import os
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_curve, auc, accuracy_score

# =========================================
# Dataset for Ink Mismatch
# =========================================
class HSIPairDatasetInk(Dataset):
    def __init__(self, pair_csv, tensor_dir):
        self.df = pd.read_csv(pair_csv)
        self.tensor_dir = tensor_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        A = torch.load(os.path.join(self.tensor_dir, row['tensor_A']), weights_only=True)
        B = torch.load(os.path.join(self.tensor_dir, row['tensor_B']), weights_only=True)

        # ✅ IMPORTANT: Ink label
        label = torch.tensor(row['same_pen'], dtype=torch.float32)

        return A, B, label


# =========================================
# CNN-LSTM Model (Same as Training)
# =========================================
class HSI_CNN_LSTM(nn.Module):
    def __init__(self, bands=149, lstm_hidden=128):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.lstm = nn.LSTM(32, lstm_hidden, batch_first=True)
        self.fc = nn.Linear(lstm_hidden, 64)

    def forward(self, x):
        B, bands, H, W = x.shape
        features = []

        for b in range(bands):
            f = self.cnn(x[:, b:b+1])
            features.append(f.view(B, -1))

        seq = torch.stack(features, dim=1)
        lstm_out, _ = self.lstm(seq)

        return self.fc(lstm_out[:, -1])


# =========================================
# Paths
# =========================================
PAIR_CSV   = r"D:\PEC\HSI_Project\processed\pair_index.csv"
TENSOR_DIR = r"D:\PEC\HSI_Project\tensors\sentences"
MODEL_PATH = r"D:\PEC\HSI_Project\checkpoints\day3_step3\hsi_cnn_lstm_contrastive.pt"

SAVE_DIR = r"D:\PEC\HSI_Project\results\day3_step5\ink_mismatch"
os.makedirs(SAVE_DIR, exist_ok=True)


# =========================================
# Load Model
# =========================================
device = "cuda" if torch.cuda.is_available() else "cpu"

model = HSI_CNN_LSTM().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

print("✅ Model loaded for Ink Mismatch Evaluation")


# =========================================
# DataLoader
# =========================================
dataset = HSIPairDatasetInk(PAIR_CSV, TENSOR_DIR)
loader = DataLoader(dataset, batch_size=1, shuffle=False)


# =========================================
# Compute Embeddings & Distances
# =========================================
all_distances = []
all_labels = []

with torch.no_grad():
    for A, B, label in loader:
        A = A.float().to(device)
        B = B.float().to(device)

        emb_A = model(A)
        emb_B = model(B)

        distance = torch.norm(emb_A - emb_B, dim=1)

        all_distances.append(distance.item())
        all_labels.append(label.item())

all_distances = np.array(all_distances)
all_labels = np.array(all_labels)

print("✅ Embeddings & distances computed")


# =========================================
# ROC, AUC, Optimal Threshold
# =========================================
fpr, tpr, thresholds = roc_curve(all_labels, -all_distances)
roc_auc = auc(fpr, tpr)

optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

print(f"🎯 Optimal Threshold: {optimal_threshold:.4f}")
print(f"📈 AUC: {roc_auc:.4f}")


# =========================================
# Accuracy
# =========================================
predictions = (-all_distances >= optimal_threshold).astype(int)
accuracy = accuracy_score(all_labels, predictions)

print(f"✅ Accuracy: {accuracy*100:.2f}%")


# =========================================
# ROC Curve Plot
# =========================================
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve – Ink Mismatch Detection")
plt.legend()
plt.grid(True)

roc_path = os.path.join(SAVE_DIR, "roc_curve.png")
plt.savefig(roc_path, dpi=300, bbox_inches="tight")
plt.close()

print("✅ ROC curve saved")


# =========================================
# Distance Distribution Plot
# =========================================
same = all_distances[all_labels == 1]
diff = all_distances[all_labels == 0]

plt.figure(figsize=(6,5))
plt.hist(same, bins=40, alpha=0.6, label="Same Ink")
plt.hist(diff, bins=40, alpha=0.6, label="Different Ink")
plt.axvline(-optimal_threshold, color='red', linestyle='--', label="Threshold")
plt.xlabel("Embedding Distance")
plt.ylabel("Frequency")
plt.title("Distance Distribution – Ink Mismatch")
plt.legend()
plt.grid(True)

dist_path = os.path.join(SAVE_DIR, "distance_distribution.png")
plt.savefig(dist_path, dpi=300, bbox_inches="tight")
plt.close()

print("✅ Distance distribution saved")


# =========================================
# Save Metrics
# =========================================
metrics = {
    "accuracy": float(accuracy),
    "auc": float(roc_auc),
    "optimal_threshold": float(optimal_threshold),
    "total_pairs": int(len(all_labels)),
    "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open(os.path.join(SAVE_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=4)

print("✅ Metrics saved")


# =========================================
# Save Distances
# =========================================
df = pd.DataFrame({
    "distance": all_distances,
    "same_ink": all_labels
})

df.to_csv(os.path.join(SAVE_DIR, "distances_labels.csv"), index=False)

print("✅ Distances saved")


# =========================================
# Save Metadata
# =========================================
meta = {
    "experiment": "Day 3 Step 5 – Ink Mismatch Evaluation",
    "model": "HSI CNN-LSTM (Contrastive Learning)",
    "bands": 149,
    "embedding_dim": 64,
    "batch_size_eval": 1,
    "device": device,
    "notes": "Ink mismatch detection using contrastive embeddings"
}

with open(os.path.join(SAVE_DIR, "metadata.json"), "w") as f:
    json.dump(meta, f, indent=4)

print("✅ Metadata saved")


print("\n🎉 Ink Mismatch Evaluation Completed")
print(f"📂 Saved in: {SAVE_DIR}")


C:\Users\dell\AppData\Local\Temp\ipykernel_16784\1527472303.py:89: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location=de

✅ Model loaded for Ink Mismatch Evaluation
✅ Embeddings & distances computed
🎯 Optimal Threshold: -0.0292
📈 AUC: 0.4683
✅ Accuracy: 73.48%
✅ ROC curve saved
✅ Distance distribution saved
✅ Metrics saved
✅ Distances saved
✅ Metadata saved

🎉 Ink Mismatch Evaluation Completed
📂 Saved in: D:\PEC\HSI_Project\results\day3_step5\ink_mismatch
